In [1]:
import jfr_processor.profilelib.jsonjfr.*

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[1], line 1, column 8: Unresolved reference: jfr_processor

In [2]:
val native_json = load_jfr_as_json("/home/Martin.Zimen/ktor_profiling/native_util_r1k.jfr")
val jvm_json = load_jfr_as_json("/home/Martin.Zimen/ktor_profiling/jvm_util_r1k.jfr")
val native_samples = process_jfr_data(native_json, ::native_process_frame)
val jvm_samples = process_jfr_data(jvm_json, ::jvm_process_frame)
val native_functions = list_functions(native_samples)
val jvm_functions = list_functions(jvm_samples)

val full_intersect = native_functions intersect jvm_functions
val only_name_intersect =
    native_functions.map { it.split('.').last() } intersect jvm_functions.map { it.split('.').last() }

In [3]:
println(full_intersect)
println(only_name_intersect)
println()
println(native_functions.filter { f -> only_name_intersect.any { f.split('.').last() == it } })
println()
println(jvm_functions.filter { f -> only_name_intersect.any { f.split('.').last() == it } })


[kotlin/coroutines/CombinedContext.get, kotlin/coroutines/CombinedContext.minusKey]
[invoke, hashCode, resumeWith, toString, <init>, addAll, releaseIntercepted, fold, plus, get, minusKey, intercepted, checkResultIsFailure, assertTrue, hasNext, assertEquals, listOf]

[kotlin/Function2.invoke, kotlin/Function0.invoke, kotlin/Any.hashCode, kotlin/coroutines/Continuation.resumeWith, kotlin/Any.toString, kotlin/collections/ArrayList.<init>, kotlin/coroutines/native/internal/BaseContinuationImpl.resumeWith, kotlin/collections/ArrayList.addAll, kotlin/test/Asserter/Asserter$assertEquals$1.invoke, kotlin/collections/ArrayList.toString, kotlin/coroutines/native/internal/ContinuationImpl.releaseIntercepted, kotlin/coroutines/native/internal/BaseContinuationImpl.releaseIntercepted, kotlin/Throwable.<init>, kotlin/Exception.<init>, kotlin/collections/HashMap.<init>, kotlin/RuntimeException.<init>, kotlin/coroutines/CoroutineContext/Element.fold, kotlin/coroutines/CoroutineContext.plus, kotlin/coro

Preliminary name matching results are terrible. There are only two successfully matched functions. When allowing more
vague matching using only the function name, there is a bigger intersection. However, after manual inspection of the
full names, these cannot be really matched together as they point to target-specific implementation.

There may be two reasons
* Inlining problem is so widespread that there are basically no functions outside that we could compare
* We used `ktor-utils` module. This module doesn't have big integration tests but smaller tests that are repeated naively without JMH. Possibly, all interesting functions are optimized out.